In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
raw_file = Path(
    r"C:\Users\aleks\Documents\GitHub\asymmetric-catalysis-qspr-ml"
    r"\thia-Michael_regression\third_split\Kfold\substrate"
    r"\no_chemberta_Kfold_Review_1\Results\ReviewerValidation"
    r"\RS42_preselected_n7_preselected_False_"
    r"DecisionTree_test_prediction_uncertainty_raw.csv"
)

raw_df = pd.read_csv(raw_file)

raw_df.head()

,bootstrap_iteration,test_11,test_13,test_41,test_52,test_59,test_31,test_18,test_27,test_29,test_56
0,1,74.0,80.0,74.0,26.0,74.0,46.0,94.0,46.0,99.0,6.0
1,2,92.0,80.0,94.5,5.0,46.0,87.0,94.5,87.0,48.0,5.0
2,3,72.0,72.0,53.0,22.0,53.0,70.0,97.0,97.0,97.0,22.0
3,4,80.0,80.0,26.5,0.0,88.0,26.5,92.5,26.5,92.5,0.0
4,5,80.0,80.0,77.0,5.0,60.0,89.0,97.0,89.0,6.0,5.0


In [4]:
print("Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
display(raw_df.head(3))

Shape: (100, 11)
Columns: ['bootstrap_iteration', 'test_11', 'test_13', 'test_41', 'test_52', 'test_59', 'test_31', 'test_18', 'test_27', 'test_29', 'test_56']


,bootstrap_iteration,test_11,test_13,test_41,test_52,test_59,test_31,test_18,test_27,test_29,test_56
0,1,74.0,80.0,74.0,26.0,74.0,46.0,94.0,46.0,99.0,6.0
1,2,92.0,80.0,94.5,5.0,46.0,87.0,94.5,87.0,48.0,5.0
2,3,72.0,72.0,53.0,22.0,53.0,70.0,97.0,97.0,97.0,22.0


In [9]:
display(raw_df)

,bootstrap_iteration,test_11,test_13,test_41,test_52,test_59,test_31,test_18,test_27,test_29,test_56
0,1,74.0,80.0,74.0,26.0,74.0,46.0,94.0,46.0,99.0,6.0
1,2,92.0,80.0,94.5,5.0,46.0,87.0,94.5,87.0,48.0,5.0
2,3,72.0,72.0,53.0,22.0,53.0,70.0,97.0,97.0,97.0,22.0
3,4,80.0,80.0,26.5,0.0,88.0,26.5,92.5,26.5,92.5,0.0
4,5,80.0,80.0,77.0,5.0,60.0,89.0,97.0,89.0,6.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...
95,96,72.0,93.0,52.0,10.0,46.0,46.0,80.0,46.0,80.0,74.0
96,97,95.0,95.0,60.0,0.0,60.0,82.0,74.0,60.0,92.0,0.0
97,98,92.0,72.0,82.0,10.0,82.0,46.0,87.0,82.0,87.0,0.0
98,99,92.0,80.0,26.0,36.0,50.0,70.0,95.0,80.0,97.0,36.0


In [5]:
prediction_columns = [
    col for col in raw_df.columns
    if col != "bootstrap_iteration"
]

summary_rows = []

for col in prediction_columns:
    predictions = pd.to_numeric(raw_df[col], errors="coerce").dropna()

    ci_lower = predictions.quantile(0.025)
    median = predictions.quantile(0.50)
    ci_upper = predictions.quantile(0.975)

    summary_rows.append({
        "test_index": col.replace("test_", ""),
        "bootstrap_prediction_median": median,
        "bootstrap_ci95_lower": ci_lower,
        "bootstrap_ci95_upper": ci_upper,
        "bootstrap_ci95_width": ci_upper - ci_lower,
        "bootstrap_prediction_sd": predictions.std(ddof=0),
        "n_bootstrap": len(predictions),
    })

bootstrap_summary = pd.DataFrame(summary_rows)

bootstrap_summary

,test_index,bootstrap_prediction_median,bootstrap_ci95_lower,bootstrap_ci95_upper,bootstrap_ci95_width,bootstrap_prediction_sd,n_bootstrap
0,11,92.00,70.0,96.25000,26.25000,9.925301,100
1,13,86.75,70.0,97.26250,27.26250,9.902746,100
2,41,58.00,26.0,97.26250,71.26250,25.222555,100
3,52,12.00,0.0,67.35000,67.35000,16.281661,100
4,59,50.00,26.0,88.00000,62.00000,16.647525,100
5,31,65.00,13.6,97.76250,84.16250,26.018188,100
6,18,92.00,3.0,97.38125,94.38125,21.030396,100
7,27,76.25,15.5,97.76250,82.26250,25.749296,100
8,29,80.00,3.0,97.52500,94.52500,35.135068,100
9,56,10.00,0.0,74.00000,74.00000,18.833391,100


In [6]:
bootstrap_summary_display = bootstrap_summary.copy()

for column in [
    "bootstrap_prediction_median",
    "bootstrap_ci95_lower",
    "bootstrap_ci95_upper",
    "bootstrap_ci95_width",
    "bootstrap_prediction_sd",
]:
    bootstrap_summary_display[column] = (
        bootstrap_summary_display[column].round(2)
    )

bootstrap_summary_display

,test_index,bootstrap_prediction_median,bootstrap_ci95_lower,bootstrap_ci95_upper,bootstrap_ci95_width,bootstrap_prediction_sd,n_bootstrap
0,11,92.00,70.0,96.25,26.25,9.93,100
1,13,86.75,70.0,97.26,27.26,9.90,100
2,41,58.00,26.0,97.26,71.26,25.22,100
3,52,12.00,0.0,67.35,67.35,16.28,100
4,59,50.00,26.0,88.00,62.00,16.65,100
5,31,65.00,13.6,97.76,84.16,26.02,100
6,18,92.00,3.0,97.38,94.38,21.03,100
7,27,76.25,15.5,97.76,82.26,25.75,100
8,29,80.00,3.0,97.52,94.52,35.14,100
9,56,10.00,0.0,74.00,74.00,18.83,100
